In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
import matplotlib.pyplot as plt
# seaborn is a better tool for this with more examples 
import seaborn as sns


df_melanoma = pd.read_csv('melanoma_mitoses.csv', sep=';')

In [2]:
df_melanoma.head()

In [3]:
df_melanoma.shape

In [4]:
df_melanoma.columns

## Missing values analysis
Missing Completely at Random (MCAR) — the missingness has no relationship to anything. Rare in practice.
Missing at Random (MAR) — the missingness depends on other observed variables. For example, Clark level might be missing more often for thicker tumours because clinicians didn't bother recording it when Breslow was clearly the more relevant measure.
Missing Not at Random (MNAR) — the missingness depends on the missing value itself. For example, if regressione is missing because pathologists only record it when they see it, then the nulls might actually be informative — they could effectively mean "not observed/not present."


features with more than 15–20% missing values were excluded because imputation of patient-specific histological findings would introduce misleading information


In [5]:
# Now start to clean the data
# In order to make this model work we will need to encode
# many of the variables to make them numeric

# First we can drop unnecessary columns

df_melanoma_X = df_melanoma.drop(columns=[ 'survival',   # leakage: another way of showing status the variable we are trying to predict
                                           'ID_prog',    # row identifier, no predictive value
                                           'clark',      # this is an alternative measure of tumour depth similiar to breslow. Data has many nulls so will use breslow instead
                                           'regressione', # there are over 500 nulls in 1804 rows in the testing data. Regression in patients is independent so imputation isn't valid
                                           'growth',      # same argument as for regression
                                           'year'])      # not needed

# ORDINAL ENCODINGS 
# This for the variables where we care about the order
# so e.g. the larger the breslow_map (thicker the tumor) 
# the more dangerous it is, so is encoded as 5

tnm_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
df_melanoma_X['TNM_stage'] = df_melanoma_X['TNM_stage'].map(tnm_map)

pt_map = {'pT1': 1, 'pT2': 2, 'pT3': 3, 'pT4': 4}
df_melanoma_X['pT'] = df_melanoma_X['pT'].map(pt_map)

breslow_map = {'< 0.76': 1, '0.76-1.50': 2, '1.51-3.99': 3, '>=4': 4}
df_melanoma_X['breslow'] = df_melanoma_X['breslow'].map(breslow_map)

# clark_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5}
# df_melanoma_X['clark'] = df_melanoma_X['clark'].map(clark_map)

# BINARY ENCODINGS
# Simple categorization between 2 options

df_melanoma_X['sex']         = df_melanoma_X['sex'].map({'F': 0, 'M': 1})
df_melanoma_X['ulcerazione'] = df_melanoma_X['ulcerazione'].map({'Absent': 0, 'Present': 1})
df_melanoma_X['til']         = df_melanoma_X['til'].map({'Absent': 0, 'Present': 1})
# df_melanoma_X['growth']      = df_melanoma_X['growth'].map({'Radial': 0, 'Vertical': 1})
# df_melanoma_X['regressione'] = df_melanoma_X['regressione'].map({'Absent': 0, 'Present': 1})
df_melanoma_X['pM']          = df_melanoma_X['pM'].map({'M0': 0, 'M1': 1})
df_melanoma_X['positive_slnb'] = df_melanoma_X['positive_slnb'].map({'0': 0, '1': 1, '>1': 2})

# ORDINAL pN 
pn_map = {'N0': 0, 'N1a': 1, 'N1b': 1, 'N1c': 1,
           'N2a': 2, 'N2b': 2, 'N2c': 2,
           'N3':  3, 'N3c': 3}
df_melanoma_X['pN'] = df_melanoma_X['pN'].map(pn_map)

# ONE-HOT ENCODE 
# Columns we need to encode but don't care about the order
# This is a standard method of working with this kind of data
# drop_first=True drops one dummy per group to avoid multicollinearity
df_melanoma_X = pd.get_dummies(df_melanoma_X, columns=['histology', 'site'], drop_first=True)

# LOG-TRANSFORM 
# This is the recommended way where you have a wide range of values 
# mitosis runs from 0 to 55 and is very skewed to the lower end of 
# that range
df_melanoma_X['mitoses_log'] = np.log1p(df_melanoma_X['mitoses'])

# ... and drop the original mitosis column
df_melanoma_X = df_melanoma_X.drop(columns=['mitoses'])

In [6]:
df_melanoma_X.columns

In [7]:
df_melanoma_X.head()
df_melanoma_X.shape

In [8]:
#### We have to check for nulls in the values to make sure we can run the model.

In [9]:
#rows, cols = df_melanoma_X.isnull().values.nonzero()
#for r, c in zip(rows, cols):
#    print(f"Row {r}, Column: {df_melanoma_X.columns[c]}")
df_melanoma_X.isnull().sum()

#import seaborn as sns
#import matplotlib.pyplot as plt

#sns.heatmap(df_melanoma_X.isnull(), yticklabels=False, cbar=False, cmap='viridis')
#plt.title('Missing Value Map')
#plt.show()

Given imputation doesn't really work in this case as each row specifies a patient, and there is no valid reason to be able to impute between 2 adjacent rows.

Check how many rows we'd lose by dropping all nulls and whether nulls overlapCheck how many rows we'd lose by dropping all nulls and whether nulls overlapThat's a very clean result. You'd lose 189 rows, leaving you with 2,066 patients — that's only an 8% reduction. Plenty of data.

The interesting finding is that 133 of those 189 rows are cases where TIL is the only missing value. So TIL is driving most of the row loss. The other columns (pT, pN, pM, breslow, ulcerazione, site, TNM_stage) contribute relatively few additional drops because their nulls are small in number and presumably overlap with each other somewhat (the staging columns especially — if pT is missing, TNM_stage is likely missing too).

Dropping rows is the right call here. You keep 2,066 completely clean rows with no imputed values, which is more than enough for logistic regression. And your justification is straightforward: all missing values represent unrecorded patient-specific clinical observations where imputation would be clinically meaningless.
If you wanted to be really thorough in your write-up, you could note that you checked whether the dropped rows differ systematically from the kept rows (e.g., are the dropped patients disproportionately one stage or one year?) — but for a 500-word write-up that's probably more detail than you need.

In [10]:
df_melanoma_X.shape

In [11]:
# drop rows that contain a null value
df_melanoma_X = df_melanoma_X.dropna()
df_melanoma_X.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Prepare data
# X here is the training data which must be a table, so 2D. Here is a good explaination..
#The 1D vs 2D distinction is purely about shape:
#  - A Series [1, 2, 3] has shape (3,) — 1D
#  - A DataFrame with one column has shape (3, 1) — 2D
#  - A DataFrame with two columns has shape (3, 2) — 2D

#  So in your case, Season is not the row identifier — it's a feature just like Operator. The row index is separate and
#  implicit. sklearn doesn't use it at all; it just sees a table of values where each row is one observation and each
#  column is one feature.
#  so we could have X = df_new[['Season', 'Operator', 'Locomotive'.....]] we are just creating a table.

X = df_melanoma_X

# only if you need to encode your categories
# ski-kitlearn can't cope with catagories such as "spring" so we have to encode them into 
# expand your 2-column X into ~27 binary columns (4 seasons + 24 operators — minus one each to avoid
# redundancy, a concept called the dummy variable trap). get_dummies handles this with drop_first=True:
# X_encoded = pd.get_dummies(X, columns=['Season', 'Operator'], drop_first=True)


# for Y that must be a series (aka a list) which is expected for target variable.
y = df_melanoma['status']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Fit logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# View learned parameters
print("Learned parameters:")
print(f"  Intercept (β₀): {model.intercept_[0]:.4f}")
print(f"  Coefficient (β₁): {model.coef_[0][0]:.4f}")
print(f"\nDecision boundary: {-model.intercept_[0]/model.coef_[0][0]:.2f}")

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Late', 'Ontime']))